# Week 84, Day 4: A neural network from scratch

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w084d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x = np.array([2.0, -1.0])           # two inputs
w = np.array([0.8, 0.5])            # their weights
b = -0.6                            # the bias
z = x @ w + b
print(round(float(z), 2), round(float(sigmoid(z)), 3))
print(sigmoid(np.array([-5.0, 0.0, 5.0])).round(3).tolist())

In [ ]:
import numpy as np

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
xor = np.array([0, 1, 1, 0])
relu = lambda z: np.maximum(z, 0)

# Two hidden neurons, chosen by hand: h1 fires for "at least one", h2 for "both".
W1 = np.array([[1.0, 1.0], [1.0, 1.0]])
b1 = np.array([0.0, -1.0])
h = relu(X @ W1 + b1)
print(h.tolist())
out = h @ np.array([1.0, -2.0])     # "at least one" minus twice "both"
print(out.tolist(), (out > 0.5).astype(int).tolist() == xor.tolist())

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
n_in, n_hidden = 2, 4
params = {"W1": rng.normal(0, 1 / np.sqrt(n_in), (n_in, n_hidden)), "b1": np.zeros(n_hidden),
          "W2": rng.normal(0, 1 / np.sqrt(n_hidden), (n_hidden, 1)), "b2": np.zeros(1)}
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
Z1 = X @ params["W1"] + params["b1"]
H = np.maximum(Z1, 0)
Z2 = H @ params["W2"] + params["b2"]
p = 1 / (1 + np.exp(-Z2.ravel()))
print(Z1.shape, H.shape, Z2.shape, p.shape)
print(sum(v.size for v in params.values()), "parameters")

In [ ]:
import numpy as np

def bce(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())

y = np.array([1, 0, 1, 0])
print(round(bce(y, np.array([0.9, 0.1, 0.8, 0.3])), 3))
print(round(bce(y, np.full(4, 0.5)), 3))
print(round(bce(y, np.array([0.01, 0.1, 0.8, 0.3])), 3))
print(round(bce(np.array([1]), np.array([0.0])), 1))

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
X = rng.normal(size=(5, 3)); y = np.array([1, 0, 1, 1, 0])
w = rng.normal(size=3)

def loss(w):
    p = 1 / (1 + np.exp(-(X @ w)))
    return -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()

p = 1 / (1 + np.exp(-(X @ w)))
analytic = X.T @ ((p - y) / len(y))
h = 1e-5
numeric = np.array([(loss(w + h * e) - loss(w - h * e)) / (2 * h) for e in np.eye(3)])
print(analytic.round(6).tolist())
print(numeric.round(6).tolist())
print(float(np.abs(analytic - numeric).max()) < 1e-8)

In [ ]:
import numpy as np

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([0, 1, 1, 0], dtype=float)
rng = np.random.default_rng(0)
W1, b1 = rng.normal(0, 1, (2, 4)), np.zeros(4)
W2, b2 = rng.normal(0, 0.5, (4, 1)), np.zeros(1)
for epoch in range(2001):
    Z1 = X @ W1 + b1; H = np.maximum(Z1, 0)
    p = (1 / (1 + np.exp(-(H @ W2 + b2)))).ravel()
    if epoch % 500 == 0:
        loss = -(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12)).mean()
        print(epoch, round(float(loss), 4), (p >= 0.5).astype(int).tolist())
    dZ2 = ((p - y) / 4)[:, None]
    dW2, db2 = H.T @ dZ2, dZ2.sum(axis=0)
    dZ1 = (dZ2 @ W2.T) * (Z1 > 0)
    dW1, db1 = X.T @ dZ1, dZ1.sum(axis=0)
    W1 -= 0.5 * dW1; b1 -= 0.5 * db1; W2 -= 0.5 * dW2; b2 -= 0.5 * db2

> **Key idea.** A neuron computes `activation(x·w + b)`; one sigmoid neuron is logistic regression and can only draw a straight boundary, so it can't learn XOR. Hidden layers with a non-linear activation (ReLU, `max(0, z)`) can bend the boundary. A layer is a matrix multiplication: `(n, inputs) @ (inputs, hidden)`, plus a bias; the forward pass runs input to output and keeps the intermediate arrays. Initialise weights randomly (and seeded) to break symmetry. Binary cross-entropy, `−mean(y log p + (1 − y) log(1 − p))`, punishes confident mistakes; 0.693 is the "always 0.5" score; clip before taking logs. Backpropagation applies the chain rule backwards: `dZ2 = (p − y)/n`, `dW2 = H.T @ dZ2`, `dZ1 = (dZ2 @ W2.T) * (Z1 > 0)`, `dW1 = X.T @ dZ1`; every gradient has its parameter's shape; check with central-difference numerical gradients. Train by repeating forward, loss, backward, update, over epochs and shuffled mini-batches, watching the loss fall.

### Exercise: Activations and their slopes

Write `sigmoid(z)` that never overflows (clip `z` to the range −500 to 500 first), `sigmoid_slope(z)` (its derivative, `s × (1 − s)`), `relu(z)` and `relu_slope(z)` (1 where `z > 0`, else 0, as floats). All take and return NumPy arrays.

In [ ]:
import numpy as np

def sigmoid(z):
    ___

def sigmoid_slope(z):
    ___

def relu(z):
    ___

def relu_slope(z):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import warnings\nimport numpy as np\n\ndef test_activations():\n    \"values and slopes\"\n    z = np.array([-2.0, 0.0, 3.0])\n    assert sigmoid(z).round(4).tolist() == [0.1192, 0.5, 0.9526]\n    assert sigmoid_slope(np.array([0.0])).tolist() == [0.25]\n    assert relu(z).tolist() == [0.0, 0.0, 3.0] and relu_slope(z).tolist() == [0.0, 0.0, 1.0]\n    with warnings.catch_warnings():\n        warnings.simplefilter(\"error\")\n        low, high = sigmoid(np.array([-1000.0, 1000.0])).tolist()\n        assert low < 1e-200 and high == 1.0, \"no overflow\"\n    h = 1e-6\n    numeric = (sigmoid(np.array([0.7]) + h) - sigmoid(np.array([0.7]) - h)) / (2 * h)\n    assert abs(float(numeric[0]) - float(sigmoid_slope(np.array([0.7]))[0])) < 1e-8")

### Exercise: The forward pass

Write `init(n_in, n_hidden, rng)`: a dict with `W1` drawn as `rng.normal(0, 1, (n_in, n_hidden))`, then `W2` as `rng.normal(0, 1, (n_hidden, 1)) / np.sqrt(n_hidden)` (in that order), and zero biases `b1` (shape `(n_hidden,)`) and `b2` (shape `(1,)`). Then `forward(params, X)`: return `(p, cache)`, where `p` is the one-dimensional array of output probabilities (ReLU hidden layer, sigmoid output) and `cache` is a dict holding `Z1`, `H` and `p`.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def init(n_in, n_hidden, rng):
    ___

def forward(params, X):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\n\ndef test_forward():\n    \"shapes line up\"\n    params = init(3, 5, np.random.default_rng(0))\n    assert params[\"W1\"].shape == (3, 5) and params[\"W2\"].shape == (5, 1)\n    assert params[\"b1\"].shape == (5,) and params[\"b2\"].shape == (1,) and not params[\"b1\"].any()\n    rng = np.random.default_rng(0)\n    assert np.allclose(params[\"W1\"], rng.normal(0, 1, (3, 5))), \"draw W1 first, as described\"\n    X = np.ones((4, 3))\n    p, cache = forward(params, X)\n    assert p.shape == (4,) and cache[\"Z1\"].shape == (4, 5) and cache[\"H\"].shape == (4, 5)\n    assert (cache[\"H\"] >= 0).all() and ((p > 0) & (p < 1)).all()\n    by_hand = 1 / (1 + np.exp(-(np.maximum(X @ params[\"W1\"], 0) @ params[\"W2\"]).ravel()))\n    assert np.allclose(p, by_hand)")

### Exercise: Cross-entropy

Write `bce(y, p, eps=1e-12)`: binary cross-entropy as a float, clipping `p` to `[eps, 1 − eps]` first. Then `accuracy(y, p)`: the fraction of examples where `p >= 0.5` matches the label, and `beats_coin(y, p)`: `True` if the loss is below that of always predicting 0.5.

In [ ]:
import numpy as np

def bce(y, p, eps=1e-12):
    ___

def accuracy(y, p):
    ___

def beats_coin(y, p):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\n\ndef test_loss():\n    \"confident mistakes cost most\"\n    y = np.array([1, 0, 1, 0])\n    assert round(bce(y, np.array([0.9, 0.1, 0.8, 0.3])), 3) == 0.198\n    assert round(bce(y, np.full(4, 0.5)), 3) == 0.693\n    assert round(bce(np.array([1]), np.array([0.0])), 1) == 27.6, \"clipped, not infinite\"\n    assert accuracy(y, np.array([0.9, 0.6, 0.8, 0.3])) == 0.75\n    assert beats_coin(y, np.array([0.9, 0.1, 0.8, 0.3])) and not beats_coin(y, np.array([0.4, 0.6, 0.4, 0.6]))")

### Exercise: Checking gradients numerically

Write `numerical_gradient(f, w, h=1e-5)`: for a function `f` of an array `w` returning a number, the array of central differences `(f(w + h) − f(w − h)) / 2h`, nudging **one element at a time** (put each element back afterwards). Then `relative_error(a, b)`: `max|a − b| / max(max|a| + max|b|, 1e-12)`, which should be tiny (below `1e-6`) when two gradients agree.

In [ ]:
import numpy as np

def numerical_gradient(f, w, h=1e-5):
    ___

def relative_error(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\n\ndef test_numerical():\n    \"nudge, measure, restore\"\n    w = np.array([[1.0, 2.0], [3.0, -1.0]])\n    g = numerical_gradient(lambda a: float((a ** 2).sum()), w)\n    assert g.shape == w.shape and np.allclose(g, 2 * w, atol=1e-6)\n    assert w.tolist() == [[1.0, 2.0], [3.0, -1.0]], \"put every element back\"\n    g = numerical_gradient(lambda a: float(np.sin(a).sum()), np.array([0.0, 1.0]))\n    assert np.allclose(g, np.cos([0.0, 1.0]), atol=1e-8)\n    assert relative_error(np.array([1.0, 2.0]), np.array([1.0, 2.0000001])) < 1e-6\n    assert relative_error(np.array([1.0]), np.array([2.0])) > 0.3")

### Exercise: Backpropagation

Using the given `forward`, write `backward(params, cache, X, y)`: the gradients of the mean binary cross-entropy with respect to `W1`, `b1`, `W2` and `b2`, as a dict of arrays with the same shapes as the parameters (follow the formulas in the lesson).

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def forward(params, X):
    Z1 = X @ params["W1"] + params["b1"]
    H = np.maximum(Z1, 0)
    p = sigmoid(H @ params["W2"] + params["b2"]).ravel()
    return p, {"Z1": Z1, "H": H, "p": p}

def backward(params, cache, X, y):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\n\ndef _bce(params, X, y):\n    p = np.clip(forward(params, X)[0], 1e-12, 1 - 1e-12)\n    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())\n\ndef test_backward():\n    \"agrees with numerical gradients\"\n    rng = np.random.default_rng(3)\n    params = {\"W1\": rng.normal(size=(3, 4)), \"b1\": rng.normal(size=4) * 0.1,\n              \"W2\": rng.normal(size=(4, 1)), \"b2\": np.array([0.1])}\n    X = rng.normal(size=(6, 3))\n    y = np.array([1, 0, 1, 1, 0, 0], dtype=float)\n    grads = backward(params, forward(params, X)[1], X, y)\n    for name in params:\n        assert grads[name].shape == params[name].shape, name\n        numeric = np.zeros_like(params[name])\n        for idx in np.ndindex(params[name].shape):\n            old = params[name][idx]\n            params[name][idx] = old + 1e-5; up = _bce(params, X, y)\n            params[name][idx] = old - 1e-5; down = _bce(params, X, y)\n            params[name][idx] = old\n            numeric[idx] = (up - down) / 2e-5\n        err = np.abs(numeric - grads[name]).max() / (np.abs(numeric).max() + np.abs(grads[name]).max())\n        assert err < 1e-6, f\"{name} disagrees: {err}\"")

### Exercise: Training on XOR

Using the given `init`, `forward` and `backward`, write `step(params, grads, rate)`: a **new** dict of parameters, each moved by `−rate × gradient` (don't change `params`). Then `train_xor(seed, epochs=2000, rate=0.5, hidden=4)`: initialise with `np.random.default_rng(seed)`, train on the four XOR examples with full-batch gradient descent, and return `(params, losses)`, where `losses` holds the cross-entropy before each step.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def init(n_in, n_hidden, rng):
    W1 = rng.normal(0, 1, (n_in, n_hidden))
    W2 = rng.normal(0, 1, (n_hidden, 1)) / np.sqrt(n_hidden)
    return {"W1": W1, "b1": np.zeros(n_hidden), "W2": W2, "b2": np.zeros(1)}

def forward(params, X):
    Z1 = X @ params["W1"] + params["b1"]
    H = np.maximum(Z1, 0)
    p = sigmoid(H @ params["W2"] + params["b2"]).ravel()
    return p, {"Z1": Z1, "H": H, "p": p}

def backward(params, cache, X, y):
    dZ2 = ((cache["p"] - y) / len(y))[:, None]
    dZ1 = (dZ2 @ params["W2"].T) * (cache["Z1"] > 0)
    return {"W1": X.T @ dZ1, "b1": dZ1.sum(axis=0), "W2": cache["H"].T @ dZ2, "b2": dZ2.sum(axis=0)}

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
Y = np.array([0, 1, 1, 0], dtype=float)

def bce(y, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())

def step(params, grads, rate):
    ___

def train_xor(seed, epochs=2000, rate=0.5, hidden=4):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\n\ndef test_train():\n    \"a network learns what a neuron can't\"\n    params = init(2, 4, np.random.default_rng(0))\n    before = {k: v.copy() for k, v in params.items()}\n    new = step(params, {k: np.ones_like(v) for k, v in params.items()}, 0.1)\n    assert all(np.array_equal(params[k], before[k]) for k in params), \"don't change params\"\n    assert np.allclose(new[\"W1\"], before[\"W1\"] - 0.1)\n    solved = 0\n    for seed in range(6):\n        trained, losses = train_xor(seed)\n        assert len(losses) == 2000 and losses[-1] < losses[0]\n        p = forward(trained, X)[0]\n        solved += (p >= 0.5).astype(int).tolist() == [0, 1, 1, 0]\n    assert solved >= 4, f\"only {solved} of 6 seeds learned XOR\"")

### Bug hunt: Overflow in the sigmoid

For very negative inputs, `np.exp(-z)` overflows and NumPy prints a `RuntimeWarning` before the answer. Clip `z` first so the program prints only `[0.0, 0.5, 1.0]`.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print(sigmoid(np.array([-1000.0, 0.0, 1000.0])).round(3).tolist())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"no warning\"\n    expect_output(\"[0.0, 0.5, 1.0]\")")

### Bug hunt: Shapes that don't line up

`W1` was made with its dimensions the wrong way round, so `X @ W1` crashes with `ValueError: matmul: Input operand 1 has a mismatch`. Give `W1` the shape `(inputs, hidden)` so it prints `(4, 3)`.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
X = np.ones((4, 2))            # 4 examples, 2 inputs
W1 = rng.normal(size=(3, 2))   # meant for 3 hidden neurons
print((X @ W1).shape)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"(n, inputs) @ (inputs, hidden)\"\n    expect_output(\"(4, 3)\")")

### Bug hunt: Identical neurons

Every weight starts at the same value, so every hidden neuron computes the same thing and gets the same update: they stay identical forever, and the network can't learn XOR (it prints `[0.5, 0.5, 0.5, 0.5]`). Initialise `W1` and `W2` randomly with `rng.normal(0, 1, shape)` and `rng.normal(0, 0.5, shape)` (in that order) so it prints `[0, 1, 1, 0]`.

In [ ]:
import numpy as np

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([0, 1, 1, 0], dtype=float)
rng = np.random.default_rng(0)
W1, b1 = np.full((2, 4), 0.5), np.zeros(4)
W2, b2 = np.full((4, 1), 0.5), np.zeros(1)
for _ in range(2000):
    Z1 = X @ W1 + b1; H = np.maximum(Z1, 0)
    p = (1 / (1 + np.exp(-(H @ W2 + b2)))).ravel()
    dZ2 = ((p - y) / 4)[:, None]
    dZ1 = (dZ2 @ W2.T) * (Z1 > 0)
    W1 -= 0.5 * X.T @ dZ1; b1 -= 0.5 * dZ1.sum(axis=0)
    W2 -= 0.5 * H.T @ dZ2; b2 -= 0.5 * dZ2.sum(axis=0)
p = (1 / (1 + np.exp(-(np.maximum(X @ W1 + b1, 0) @ W2 + b2)))).ravel()
print([round(float(v), 1) for v in p] if np.allclose(p, 0.5, atol=0.05) else (p >= 0.5).astype(int).tolist())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"break the symmetry\"\n    expect_output(\"[0, 1, 1, 0]\")")

### Puzzle: One training step

Arrange one step of training: forward pass, loss, backward pass, then the update. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the shape of X @ W1 for X of shape (32, 10) and W1 of shape (10, 64), as a tuple
q = ___   # how many parameters (weights and biases) a layer from 10 inputs to 64 neurons has
r = ___   # relu applied to -3.0
s = ___   # the cross-entropy, to 3 places, of always predicting 0.5
t = ___   # True or False: a network with no non-linear activations can learn XOR

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == (32, 64) and q == 704 and r == 0.0, \"p, q and r\"\n    assert s == 0.693 and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: an MLP class

Write a class `MLP(n_in, n_hidden=16, rate=0.5, seed=0)` wrapping today's functions (given) in the scikit-learn style. It creates `self.rng = np.random.default_rng(seed)` and initialises its parameters from it, and keeps `self.history = []`. `fit(X, y, epochs=500, batch_size=None)` trains for `epochs` epochs: with no `batch_size`, one full-batch step per epoch; otherwise, each epoch visits the examples in the order `self.rng.permutation(len(X))`, in consecutive batches of `batch_size` (the last may be smaller), one step per batch. After each epoch it appends the cross-entropy on **all** of `X` to `history`. It returns `self`. `predict_proba(X)`, `predict(X)` (1 where the probability is at least 0.5) and `score(X, y)` (accuracy) work as in scikit-learn. On the two-moons data, it should beat logistic regression clearly.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def init(n_in, n_hidden, rng):
    W1 = rng.normal(0, 1, (n_in, n_hidden))
    W2 = rng.normal(0, 1, (n_hidden, 1)) / np.sqrt(n_hidden)
    return {"W1": W1, "b1": np.zeros(n_hidden), "W2": W2, "b2": np.zeros(1)}

def forward(params, X):
    Z1 = X @ params["W1"] + params["b1"]
    H = np.maximum(Z1, 0)
    p = sigmoid(H @ params["W2"] + params["b2"]).ravel()
    return p, {"Z1": Z1, "H": H, "p": p}

def backward(params, cache, X, y):
    dZ2 = ((cache["p"] - y) / len(y))[:, None]
    dZ1 = (dZ2 @ params["W2"].T) * (cache["Z1"] > 0)
    return {"W1": X.T @ dZ1, "b1": dZ1.sum(axis=0), "W2": cache["H"].T @ dZ2, "b2": dZ2.sum(axis=0)}

def bce(y, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())

class MLP:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\nfrom sklearn.datasets import make_moons\nfrom sklearn.linear_model import LogisticRegression\n\ndef _moons():\n    X, y = make_moons(300, noise=0.2, random_state=0)\n    return X[:200], y[:200], X[200:], y[200:]\n\ndef test_full_batch():\n    \"curved boundaries\"\n    X_train, y_train, X_test, y_test = _moons()\n    model = MLP(2, 16, 0.5, seed=0).fit(X_train, y_train, epochs=1000)\n    assert isinstance(model, MLP) and len(model.history) == 1000\n    assert model.history[-1] < model.history[0] / 4\n    linear = LogisticRegression().fit(X_train, y_train).score(X_test, y_test)\n    assert model.score(X_train, y_train) >= 0.95 and model.score(X_test, y_test) >= linear + 0.05\n    assert model.predict(X_test[:5]).tolist() == (model.predict_proba(X_test[:5]) >= 0.5).astype(int).tolist()\n\ndef test_mini_batches():\n    \"shuffled batches\"\n    X_train, y_train, X_test, y_test = _moons()\n    model = MLP(2, 16, 0.1, seed=1).fit(X_train, y_train, epochs=200, batch_size=20)\n    assert len(model.history) == 200 and model.score(X_test, y_test) >= 0.9\n    again = MLP(2, 16, 0.1, seed=1).fit(X_train, y_train, epochs=5, batch_size=20)\n    first = MLP(2, 16, 0.1, seed=1).fit(X_train, y_train, epochs=5, batch_size=20)\n    assert again.history == first.history, \"the same seed gives the same training\"\n    xor = MLP(2, 4, 0.5, seed=0).fit(np.array([[0, 0], [0, 1], [1, 0], [1, 1]]), np.array([0, 1, 1, 0]), epochs=2000)\n    assert xor.predict(np.array([[0, 0], [0, 1], [1, 0], [1, 1]])).tolist() == [0, 1, 1, 0]")

## Warm-up

A **neuron** multiplies each input by a weight, adds them up with a bias, and passes the total `z` through an **activation function**. The **sigmoid** squeezes any number into the range 0 to 1, so the output can be read as a probability. That's exactly logistic regression from Day 2: one neuron.

## A neuron, and what it can't do

A single neuron's decision is "is `w·x + b` above 0?", which divides the inputs with a **straight line** (a flat plane in more dimensions). Many problems aren't like that. The classic example is **XOR**: output 1 when exactly one of two inputs is 1. The points `(0, 1)` and `(1, 0)` must be on one side, `(0, 0)` and `(1, 1)` on the other, and no straight line does it. (Showing this limit, in 1969, held the field back for years.)

The fix is to put neurons in **layers**. A **hidden layer** of several neurons each draws its own line; a neuron after them combines those lines into a bent boundary. But only if each hidden neuron has a **non-linear** activation: without one, a layer of weighted sums followed by another weighted sum is still just one weighted sum, still a straight line. The most common hidden activation is **ReLU**, `max(0, z)`: zero for negative totals, unchanged for positive ones.

Hand-picked weights solve XOR with two hidden neurons. Training finds such weights by itself.

## Layers and the forward pass

With a batch of examples `X` (shape `(n, inputs)`), a whole layer is one matrix multiplication: weights `W1` of shape `(inputs, hidden)` and a bias `b1` of shape `(hidden,)` give `Z1 = X @ W1 + b1`, shape `(n, hidden)`, and `H = relu(Z1)`. The output layer does the same with `W2` of shape `(hidden, 1)`: `Z2 = H @ W2 + b2`, and `p = sigmoid(Z2)`, one probability per example. Running the input through to the output is the **forward pass**. Keep the intermediate arrays (`Z1`, `H`, `p`): the backward pass needs them.

Shapes are the main source of bugs. Read each multiplication as `(n, a) @ (a, b) → (n, b)`: the inner sizes must match.

Initial weights must be **random** and fairly small (here a normal distribution scaled by `1/√inputs`). If every weight starts equal, every hidden neuron computes the same thing, gets the same gradient and stays identical forever; the randomness **breaks the symmetry**. Seed the generator so runs can be repeated.

## Cross-entropy

To train a classifier that outputs probabilities, the loss should punish confident mistakes hard. **Binary cross-entropy** does: `loss = −mean(y·log p + (1 − y)·log(1 − p))`. Predicting 0.9 for a true 1 costs `−log 0.9 ≈ 0.105`; predicting 0.01 costs `−log 0.01 ≈ 4.6`. A model that always says 0.5 scores `log 2 ≈ 0.693`, which is the number to beat. Clip `p` slightly away from 0 and 1 before taking logs, so a perfectly confident wrong answer gives a large number instead of infinity.

## Backpropagation

Gradient descent needs the gradient of the loss with respect to **every** weight. The network is a chain of simple steps, so the **chain rule** gives them, working backwards from the loss (hence **backpropagation**). For sigmoid output with cross-entropy, the first step is remarkably simple, and the rest follow:

```
dZ2 = (p − y) / n                 shape (n, 1)
dW2 = H.T @ dZ2                   shape (hidden, 1)      db2 = dZ2.sum(axis=0)
dH  = dZ2 @ W2.T                  shape (n, hidden)
dZ1 = dH * (Z1 > 0)               ReLU passes gradient only where it was active
dW1 = X.T @ dZ1                   shape (inputs, hidden) db1 = dZ1.sum(axis=0)
```

Each gradient has the same shape as the thing it's the gradient of, which is a good check. A better one is a **numerical gradient**: nudge one weight up by a tiny `h`, then down, and see how the loss changes, `(loss(w + h) − loss(w − h)) / 2h`. It's far too slow for training (two forward passes per weight), but if it agrees with your backpropagation on a small network, your formulas are right.

## Training

Training repeats four steps: **forward** (predictions), **loss** (how wrong), **backward** (gradients), **update** (`W -= rate × dW` for every parameter). One pass through the training data is an **epoch**. With large data sets, each update uses a small random **mini-batch** of examples instead of all of them: each step is noisier but far cheaper, and the noise often helps. Shuffle the order every epoch.

Watch the loss as you train: it should fall. If it rises or jumps about, the learning rate is too high. If it gets stuck high, the network may be too small, the rate too low, or (with ReLU) some hidden neurons may have "died", stuck at zero for every example, which is one reason to try more than one seed.

It learns XOR, which no single neuron can.

## Practice

## Quiz

## Daily challenge

## Before you go

Take your `MLP` and change one thing at a time on the moons data: 2 hidden neurons instead of 16, a learning rate of 5, then 0.001, and three different seeds. Note the final loss and test accuracy for each. Which change hurts most, and why? Tomorrow is **PyTorch: a first look**: the same ideas with automatic gradients, so you never write `backward` by hand again.